In [1]:
import sys
sys.path.append("../src")

from pathlib import Path
import pandas as pd

from energy_load_forecast.ingestion import load_and_resample
from energy_load_forecast.preprocessing import clean_tail, add_calendar_features

In [2]:
series = load_and_resample(Path("../data/raw/entsoe/load_RO_2026-09-13_2026-09-20.csv"))
series_clean = clean_tail(series)
df = add_calendar_features(series_clean.to_frame())
print("Shape:", df.shape)

Shape: (143, 5)


In [3]:
df["load_mw_lag1"] = df["load_mw"].shift(1)
df["load_mw_lag24"] = df["load_mw"].shift(24)
df["load_mw_lag168"] = df["load_mw"].shift(168)

df["load_mw_rolling_mean_3h"] = df["load_mw"].shift(1).rolling(window=3).mean()
df["load_mw_rolling_mean_24h"] = df["load_mw"].shift(1).rolling(window=24).mean()

print("Head:\n", df.head(30))

Head:
                            load_mw  hour_local  day_of_week_local  \
timestamp_utc                                                       
2026-09-13 00:00:00+00:00  4417.25           3                  6   
2026-09-13 01:00:00+00:00  4406.50           4                  6   
2026-09-13 02:00:00+00:00  4544.25           5                  6   
2026-09-13 03:00:00+00:00  4611.25           6                  6   
2026-09-13 04:00:00+00:00  4719.75           7                  6   
2026-09-13 05:00:00+00:00  4751.75           8                  6   
2026-09-13 06:00:00+00:00  4602.50           9                  6   
2026-09-13 07:00:00+00:00  4330.50          10                  6   
2026-09-13 08:00:00+00:00  3989.50          11                  6   
2026-09-13 09:00:00+00:00  3831.00          12                  6   
2026-09-13 10:00:00+00:00  4035.00          13                  6   
2026-09-13 11:00:00+00:00  4153.75          14                  6   
2026-09-13 12:00:00+00:00  

In [4]:
print("Tail:\n", df.tail(10))

Tail:
                            load_mw  hour_local  day_of_week_local  \
timestamp_utc                                                       
2026-09-18 13:00:00+00:00  5192.00          16                  4   
2026-09-18 14:00:00+00:00  5784.25          17                  4   
2026-09-18 15:00:00+00:00  6223.75          18                  4   
2026-09-18 16:00:00+00:00  6583.00          19                  4   
2026-09-18 17:00:00+00:00  6720.50          20                  4   
2026-09-18 18:00:00+00:00  6396.25          21                  4   
2026-09-18 19:00:00+00:00  5852.50          22                  4   
2026-09-18 20:00:00+00:00  5415.50          23                  4   
2026-09-18 21:00:00+00:00  5110.25           0                  5   
2026-09-18 22:00:00+00:00  5072.50           1                  5   

                           is_weekend_local  month_local  load_mw_lag1  \
timestamp_utc                                                            
2026-09-18 13:00

In [5]:
check_idx = 10
expected = df["load_mw"].iloc[check_idx - 3:check_idx].mean()
actual = df["load_mw_rolling_mean_3h"].iloc[check_idx]
print("Expected:", expected, "Actual:", actual, "Match:", abs(expected - actual) < 1e-6)

Expected: 4050.3333333333335 Actual: 4050.3333333333335 Match: True


In [6]:
print("lag168 all NaN:", df["load_mw_lag168"].isna().all())

lag168 all NaN: True
